# Elliptic bursting with the resonant band-pass selector

Companion code for *Excitability Types and Bursting with a Two-Block Spiking Primitive*,
Section V-C. **This notebook produces Fig. 5.**

The high-pass selector cannot burst with a single slow state, because rest and repetitive
firing never coexist there. The band-pass carries the missing ingredient: for $k > 3/2$ the
Hopf is subcritical, so a stable cycle survives below the Hopf down to a fold of limit
cycles and rest and spiking coexist in between. One slow negative feedback closed around
that window is enough:

$$\tau_s \dot z = -z + y, \qquad u_\mathrm{eff} = u - g z, \qquad \tau_s \gg 1/f_c .$$

Three dynamic states in total, and no new nonlinearity. The two constraints of Section IV-B
carry over unchanged: the slow branch has to sense the output rather than the feedback
signal, which is zero mean in both regimes, and it has to enter at the sigmoid port, since
an input at the selector port cannot move the operating point.

In [ ]:
using Plots, LaTeXStrings, DifferentialEquations, DiffEqCallbacks
using Printf, Statistics, Plots.PlotMeasures

gr(guidefontsize = 14, tickfontsize = 12, legendfontsize = 12, margin = 5Plots.mm, grid = true)
myBlue   = RGBA(131/255, 174/255, 218/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPurple = RGBA(169/255,  90/255, 179/255, 1)
myRed    = RGBA(158/255,   3/255,   8/255, 1)
default(fmt = :png);

## The model

In [ ]:
Base.@kwdef struct BPBurster
    k::Float64    = 3.0       # sigmoid gain, k > 3/2 for a subcritical onset
    Q::Float64    = 1.0       # quality factor
    fc::Float64   = 1.0       # center frequency [Hz]
    u::Float64    = -0.78     # constant bias, at the sigmoid port
    g::Float64    = 0.55      # slow feedback gain
    taus::Float64 = 20.0      # slow time constant, taus >> 1/fc
end

omega(p::BPBurster) = 2pi * p.fc

# states: X = [x1, x2, z]
bp_w(X, p)    = (omega(p) / p.Q) * X[2]
bp_ueff(X, p) = p.u - p.g * X[3]
bp_y(X, p)    = tanh(p.k * (bp_ueff(X, p) + bp_w(X, p)))

function rhs!(dX, X, p::BPBurster, t)
    y = bp_y(X, p)
    dX[1] = X[2]
    dX[2] = -omega(p)^2 * X[1] - (omega(p) / p.Q) * X[2] + y
    dX[3] = (-X[3] + y) / p.taus
    nothing
end

function simulate(p::BPBurster; tspan = (0.0, 400.0), X0 = [0.0, -0.3, 0.0], saveat = 0.002)
    solve(ODEProblem(rhs!, X0, tspan, p), Tsit5();
          abstol = 1e-10, reltol = 1e-9, saveat = saveat)
end

function signals(sol, p::BPBurster)
    t  = sol.t
    w  = (omega(p) / p.Q) .* [s[2] for s in sol.u]
    z  = [s[3] for s in sol.u]
    ue = p.u .- p.g .* z
    return (t = t, w = w, z = z, ueff = ue, y = tanh.(p.k .* (ue .+ w)))
end

rheobase(k) = atanh(sqrt(1 - 1/k)) / k;

## Fig. 5

Fig. 5 puts the two selectors side by side, so this section adds the high-pass cascade of
Section IV-B and the measurement helpers. Nothing above is redefined. Only the switch
position $\sigma_s = x_s$ is needed here, and it makes the drive of the fast loop
continuous, so no branch re-validation is required after a slow switch.

In [ ]:
# --- the open cascade, exact hybrid integration -----------------------------

Base.@kwdef mutable struct Cascade
    ks::Float64   = 3.0
    kf::Float64   = 3.0
    taus::Float64 = 60.0      # slow time constant
    tau::Float64  = 1.0       # fast time constant
    us::Float64   = 0.0       # bias of the slow loop
    u0::Float64   = -0.55     # bias of the fast loop, negative so that rest sits low
    beta::Float64 = 0.70      # tap gain, sigma_s = x_s
    ys::Float64   = 1.0
    yf::Float64   = 1.0
end

casc_drive(p::Cascade, x) = p.u0 + p.beta * x[1]

function casc_rhs!(dx, x, p::Cascade, t)
    dx[1] = (-x[1] + p.ys) / p.taus
    dx[2] = (-x[2] + p.yf) / p.tau
    nothing
end

cond_s(x, t, integ) = (p = integ.p;
    p.ys > 0 ? x[1] - (p.us + 1 - 1/p.ks) : (p.us - 1 + 1/p.ks) - x[1])
cond_f(x, t, integ) = (p = integ.p; ue = casc_drive(p, x);
    p.yf > 0 ? x[2] - (ue + 1 - 1/p.kf) : (ue - 1 + 1/p.kf) - x[2])

flip_s!(integ) = (integ.p.ys = -integ.p.ys; nothing)
flip_f!(integ) = (integ.p.yf = -integ.p.yf; nothing)

function simulate(p::Cascade; tspan = (0.0, 1200.0), x0 = [0.2, 0.0], dtmax = 0.02)
    p.ys, p.yf = 1.0, 1.0
    sv = SavedValues(Float64, Tuple{Float64,Float64})
    cb = CallbackSet(ContinuousCallback(cond_s, flip_s!),
                     ContinuousCallback(cond_f, flip_f!),
                     SavingCallback((x, t, integ) -> (integ.p.ys, integ.p.yf), sv))
    sol = solve(ODEProblem(casc_rhs!, x0, tspan, p), Tsit5();
                callback = cb, abstol = 1e-10, reltol = 1e-9, dtmax = dtmax)
    ts = sv.t
    yf = [v[2] for v in sv.saveval]
    xs = [sol(t)[1] for t in ts]
    return (t = ts, drive = p.u0 .+ p.beta .* xs, y = yf)
end

# --- spike times ------------------------------------------------------------
"""High-pass side: end of a +1 plateau of the fast loop."""
spiketimes(t, y) = [t[i] for i in 1:length(y)-1 if y[i] > 0 && y[i+1] < 0]

"""Band-pass side: upward zero crossing of w, linearly interpolated."""
function spikes_cross(t, w)
    sp = Float64[]
    for i in 1:length(t)-1
        if w[i] < 0 && w[i+1] >= 0
            push!(sp, t[i] - w[i] * (t[i+1] - t[i]) / (w[i+1] - w[i]))
        end
    end
    sp
end

# --- burst grouping, display window, modulated quantities -------------------
function bursts(sp; gapfac = 3.0)
    length(sp) < 6 && return Vector{Vector{Float64}}()
    isi  = diff(sp)
    gaps = findall(>(gapfac * median(isi)), isi)
    isempty(gaps) && return Vector{Vector{Float64}}()
    out, st = Vector{Vector{Float64}}(), 1
    for g in gaps
        push!(out, sp[st:g]); st = g + 1
    end
    push!(out, sp[st:end])
    filter(b -> length(b) >= 3, out)
end

"""Window covering `nb` complete bursts, padded by `pad` of the silent gap."""
function window(sp, nb, fallback; pad = 0.75)
    bs = bursts(sp)
    length(bs) < nb + 2 && return fallback
    j   = length(bs) - 1                 # drop the last one, usually truncated
    i   = max(2, j - nb + 1)
    gap = bs[i][1] - bs[i-1][end]
    (bs[i][1] - pad * gap, bs[i+nb-1][end] + pad * gap)
end

"""Instantaneous rate and peak-to-peak amplitude, one point per interspike interval,
interburst intervals dropped."""
function rate_amp(t, sig, sp; gapfac = 8.0)
    tm, f, A = Float64[], Float64[], Float64[]
    length(sp) < 4 && return (tm = tm, f = f, A = A)
    isi = diff(sp)
    med = median(isi)
    idx = clamp.([searchsortedfirst(t, s) for s in sp], 1, length(t))
    for i in eachindex(isi)
        isi[i] > gapfac * med && continue
        a, b = idx[i], idx[i+1]
        b <= a && continue
        push!(tm, 0.5 * (sp[i] + sp[i+1]))
        push!(f,  1 / isi[i])
        push!(A,  maximum(sig[a:b]) - minimum(sig[a:b]))
    end
    (tm = tm, f = f, A = A)
end

# --- tick helper, shared with the other figure notebooks --------------------
fmtnum(x, d) = Printf.format(Printf.Format("%.$(d)f"), iszero(x) ? 0.0 : x)

function niceticks(lo, hi; n = 4)
    raw  = (hi - lo) / n
    mag  = 10.0^floor(log10(raw))
    r    = raw / mag
    step = (r < 1.5 ? 1.0 : r < 3.0 ? 2.0 : r < 7.0 ? 5.0 : 10.0) * mag
    v    = collect(ceil(lo/step - 1e-9)*step : step : floor(hi/step + 1e-9)*step)
    d    = max(0, Int(-floor(log10(step))))
    (v, [latexstring(fmtnum(x, d)) for x in v])
end

In [ ]:
# =====================================================================
#  Left  column : H_hp, cascade, switch on x_s, one burst per slow period
#                 -> parabolic, the RATE is modulated
#  Right column : H_r, band-pass with one slow low-pass state
#                 -> elliptic, the AMPLITUDE is modulated
# =====================================================================

NB_L, NB_R = 1, 1        # bursts shown per column
FSg, FSt, FSl, FSa = 14, 12, 11, 14

# --- run both models --------------------------------------------------------
pL   = Cascade(ks = 3.0, kf = 3.0, taus = 60.0, tau = 1.0, us = 0.0, u0 = -0.55, beta = 0.70)
L    = simulate(pL; tspan = (0.0, 1600.0))
spL  = filter(s -> s > 300.0, spiketimes(L.t, L.y))        # drop the transient
winL = window(spL, NB_L, (400.0, 700.0))

pR   = BPBurster(k = 3.0, Q = 1.0, fc = 1.0, u = -0.78, g = 0.55, taus = 20.0)
R    = signals(simulate(pR; tspan = (0.0, 600.0)), pR)
spR  = filter(s -> s > 150.0, spikes_cross(R.t, R.w))
winR = window(spR, NB_R, (200.0, 300.0))

@printf("left  window : %.1f -> %.1f tau  (%d spikes)\n",
        winL[1], winL[2], count(s -> winL[1] <= s <= winL[2], spL))
@printf("right window : %.1f -> %.1f s    (%d spikes)\n",
        winR[1], winR[2], count(s -> winR[1] <= s <= winR[2], spR))
@printf("H_r thresholds: Hopf %.3f, fold of cycles %.3f\n", -rheobase(pR.k), -0.497)

mL, mR = findall(t -> winL[1] <= t <= winL[2], L.t), findall(t -> winR[1] <= t <= winR[2], R.t)
tL, tR = L.t[mL] .- winL[1], R.t[mR] .- winR[1]
WL, WR = winL[2] - winL[1], winR[2] - winR[1]

raL = rate_amp(L.t, L.y, filter(s -> winL[1] <= s <= winL[2], spL))
raR = rate_amp(R.t, R.y, filter(s -> winR[1] <= s <= winR[2], spR))
@printf("left  : f in [%.3f, %.3f] 1/tau, A in [%.3f, %.3f]\n",
        minimum(raL.f), maximum(raL.f), minimum(raL.A), maximum(raL.A))
@printf("right : f in [%.3f, %.3f] Hz,    A in [%.3f, %.3f]\n",
        minimum(raR.f), maximum(raR.f), minimum(raR.A), maximum(raR.A))

# --- axes -------------------------------------------------------------------
trim(xt, W) = (xt[1][xt[1] .<= W], xt[2][xt[1] .<= W])
xtL, xtR = trim(niceticks(0, WL), WL), trim(niceticks(0, WR), WR)
blank(xt) = (xt[1], fill("", length(xt[1])))
ytY = ([-1.0, 0.0, 1.0], [L"-1", L"0", L"1"])
ytN = ([0.0, 0.5, 1.0], [L"0", L"0.5", L"1"])
base = (; guidefontsize = FSg, tickfontsize = FSt, legendfontsize = FSl,
          foreground_color_legend = nothing, background_color_legend = nothing)

# --- row 1: the slow drive against the thresholds ---------------------------
loL, hiL = extrema(L.drive[mL]); padL = 0.10 * (hiL - loL)
r1L = plot(tL, L.drive[mL]; lw = 3.0, lc = :black, legend = false,
           xlims = (0, WL), ylims = (loL - padL, hiL + padL), ylabel = L"u_f",
           xticks = blank(xtL), yticks = niceticks(loL, hiL; n = 3), base...)
hline!(r1L, [-1/pL.kf, 1/pL.kf]; ls = :dash, lc = myRed, lw = 2.0, label = "")
annotate!(r1L, 0.02WL, hiL + padL, text(L"H_\mathrm{hp}", FSa, :left, :top, :black))

loR, hiR = extrema(R.ueff[mR]); padR = 0.10 * (hiR - loR)
r1R = plot(tR, R.ueff[mR]; lw = 3.0, lc = :black, legend = false,
           xlims = (0, WR), ylims = (min(loR - padR, -0.51), hiR + padR),
           ylabel = L"u_{\mathrm{eff}}", xticks = blank(xtR),
           yticks = niceticks(min(loR, -0.50), hiR; n = 3), base...)
hline!(r1R, [-rheobase(pR.k)]; ls = :dash, lc = myRed,    lw = 2.0, label = "")
hline!(r1R, [-0.497];          ls = :dot,  lc = myPurple, lw = 2.0, label = "")
annotate!(r1R, 0.02WR, hiR + padR, text(L"H_\mathrm{r}", FSa, :left, :top, :black))

# --- row 2: the output, rest low and spikes up ------------------------------
r2L = plot(tL, L.y[mL]; lw = 3.0, lc = myBlue, legend = false,
           xlims = (0, WL), ylims = (-1.15, 1.15), ylabel = L"y_f",
           xticks = blank(xtL), yticks = ytY, base...)
r2R = plot(tR, R.y[mR]; lw = 3.0, lc = myBlue, legend = false,
           xlims = (0, WR), ylims = (-1.15, 1.15), ylabel = L"y",
           xticks = blank(xtR), yticks = ytY, base...)

# --- row 3: what is modulated, each quantity normalized per panel -----------
r3L = plot(; xlims = (0, WL), ylims = (-0.05, 1.15), xlabel = L"t/\tau",
           ylabel = L"f/f_{\max},\; A/A_{\max}", xticks = xtL, yticks = ytN,
           legend = :bottomright, legend_columns = 2, base...)
scatter!(r3L, raL.tm .- winL[1], raL.f ./ maximum(raL.f);
         ms = 5.0, mc = myPurple, msc = myPurple, label = L"f/f_{\max}")
scatter!(r3L, raL.tm .- winL[1], raL.A ./ maximum(raL.A);
         ms = 5.0, mc = myOrange, msc = myOrange, label = L"A/A_{\max}")

r3R = plot(; xlims = (0, WR), ylims = (-0.05, 1.15), xlabel = L"t\,f_c",
           ylabel = "", xticks = xtR, yticks = ytN, legend = false, base...)
scatter!(r3R, raR.tm .- winR[1], raR.f ./ maximum(raR.f);
         ms = 5.0, mc = myPurple, msc = myPurple, label = "")
scatter!(r3R, raR.tm .- winR[1], raR.A ./ maximum(raR.A);
         ms = 5.0, mc = myOrange, msc = myOrange, label = "")

fig5 = plot(r1L, r1R, r2L, r2R, r3L, r3R;
            layout = (3, 2), size = (1200, 560),
            margins = 0Plots.mm, left_margin = 6.1Plots.mm,
            right_margin = 2Plots.mm, top_margin = 2Plots.mm,
            bottom_margin = [0Plots.mm 0Plots.mm 0Plots.mm 0Plots.mm 6Plots.mm 6Plots.mm])

mkpath("figures")
savefig(fig5, "figures/fig5.pdf")
fig5